<a href="https://colab.research.google.com/github/jen257/CS4287_CamVid_FCN/blob/main/CS4287_Assign2_23365684_23033614_23110465.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CS4287 Assignment 1: CNNs
## Team Members:
1. HuiLin Chong (ID: 23365684)
2. JiaLiang Lyu (ID: 23033614)
3. JiaDong Chen (ID: 23110465)


## Execution Status:


## Third-Party Sources:
https://github.com/jen257/CS4287_CamVid_FCN

# Task

###Task 1: Develop Dummy Data Placeholder DataLoader

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

# Define global constants directly instead of importing from config.py
BATCH_SIZE = 8
NUM_CLASSES = 12
IMG_HEIGHT = 360
IMG_WIDTH = 480


# Define a placeholder Dataset to test the pipeline before real data is loaded
class FakeCamVidDataset(Dataset):
    def __init__(self, size=32):
        # Total number of dummy samples
        self.size = size

    def __len__(self):
        # Return the total dataset size
        return self.size

    def __getitem__(self, idx):
        # Generate random RGB image tensor with shape (3, Height, Width) in range [0, 1)
        image = torch.rand(3, IMG_HEIGHT, IMG_WIDTH, dtype=torch.float32)

        # Generate random mask with integer class labels [0, NUM_CLASSES - 1]
        # Shape: (Height, Width)
        mask = torch.randint(0, NUM_CLASSES, (IMG_HEIGHT, IMG_WIDTH), dtype=torch.long)

        return image, mask


# Instantiate dataset and dataloader for stage 1 interface testing
fake_dataset = FakeCamVidDataset(size=32)
fake_loader = DataLoader(fake_dataset, batch_size=BATCH_SIZE, shuffle=True)

# Quick sanity check: verify output shapes of one batch
if __name__ == "__main__":
    for images, masks in fake_loader:
        print("Batch Image Shape:", images.shape)  # Expected: [8, 3, 360, 480]
        print("Batch Mask Shape :", masks.shape)   # Expected: [8, 360, 480]
        break

Batch Image Shape: torch.Size([8, 3, 360, 480])
Batch Mask Shape : torch.Size([8, 360, 480])


###Task 2: Implement FCN-32s without Skip Connections and Test Forward Pass with Dummy Data

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Configuration matching config.py requirements
BATCH_SIZE = 8
NUM_CLASSES = 12
IMG_HEIGHT = 360
IMG_WIDTH = 480


class FCN32s(nn.Module):

  def __init__(self, num_classes=NUM_CLASSES):
    super(FCN32s, self).__init__()

    # Layer-by-layer VGG-style feature extractor
    self.features = nn.Sequential(
        # Block 1 (Downsampling: 1/2)
        nn.Conv2d(3, 64, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(64, 64, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=2, stride=2),
        # Block 2 (Downsampling: 1/4)
        nn.Conv2d(64, 128, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(128, 128, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=2, stride=2),
        # Block 3 (Downsampling: 1/8)
        nn.Conv2d(128, 256, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(256, 256, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=2, stride=2),
        # Block 4 (Downsampling: 1/16)
        nn.Conv2d(256, 512, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(512, 512, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=2, stride=2),
        # Block 5 (Downsampling: 1/32)
        nn.Conv2d(512, 512, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(512, 512, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=2, stride=2),
    )

    # Fully convolutional classifier layers (fc6, fc7 replaced with 1x1 conv)
    self.classifier = nn.Sequential(
        nn.Conv2d(512, 512, kernel_size=1),
        nn.ReLU(inplace=True),
        nn.Dropout2d(p=0.5),
        nn.Conv2d(512, num_classes, kernel_size=1),
    )

    # 32x upsampling using transposed convolution
    self.upsample = nn.ConvTranspose2d(
        in_channels=num_classes,
        out_channels=num_classes,
        kernel_size=32,
        stride=32,
        bias=False,
    )

  def forward(self, x):
    # Extract features (downsampled by 32)
    feat = self.features(x)

    # Predict class scores per pixel in latent space
    score = self.classifier(feat)

    # Upsample back
    out = self.upsample(score)

    # Resize/crop to strictly match input dimensions (360x480) dynamically
    out = F.interpolate(out, size=(x.shape[2], x.shape[3]), mode="bilinear", align_corners=False)

    # Return logits without softmax
    return out

# Task 2: Dummy Data Forward Pass Test
if __name__ == "__main__":
  # Initialize model
  model = FCN32s(num_classes=NUM_CLASSES)
  model.eval()

  # Create dummy input: (BATCH_SIZE, 3, IMG_HEIGHT, IMG_WIDTH)
  dummy_images = torch.randn(
      BATCH_SIZE, 3, IMG_HEIGHT, IMG_WIDTH, dtype=torch.float32
  )

  # Forward pass
  with torch.no_grad():
    output_logits = model(dummy_images)

  # Verify output tensor shape: (BATCH_SIZE, NUM_CLASSES, IMG_HEIGHT, IMG_WIDTH)
  print(f"Input shape:  {dummy_images.shape}")
  print(f"Output shape: {output_logits.shape}")

  # Check dimensions match project specification
  assert output_logits.shape == (
      BATCH_SIZE,
      NUM_CLASSES,
      IMG_HEIGHT,
      IMG_WIDTH,
  ), "Dimension mismatch!"
  print("Test passed successfully!")

Input shape:  torch.Size([8, 3, 360, 480])
Output shape: torch.Size([8, 12, 360, 480])
Test passed successfully!


###Task 3: Implement mIoU Calculation Function and Basic Training Loop, Test Backpropagation with Dummy Data

In [ ]:
import torch
import torch.nn as nn

# 1. Setup Dummy Data
BATCH_SIZE = 8
NUM_CLASSES = 12
IMG_HEIGHT = 360
IMG_WIDTH = 480

print("--- Step 1: Preparing Tensors ---")
# FCN output: (Batch, Classes, Height, Width)
fake_preds = torch.randn(BATCH_SIZE, NUM_CLASSES, IMG_HEIGHT, IMG_WIDTH, requires_grad=True)
print(f"Prediction shape: {fake_preds.shape}")

# Labels: 0 to 11 (Batch, Height, Width)
fake_labels = torch.randint(0, NUM_CLASSES, (BATCH_SIZE, IMG_HEIGHT, IMG_WIDTH))
print(f"Label shape: {fake_labels.shape}")

# 2. Test CrossEntropyLoss & Backprop
print("\n--- Step 2: Loss & Backprop ---")
loss_function = nn.CrossEntropyLoss()

# Calculate loss
loss = loss_function(fake_preds, fake_labels)
print(f"Loss: {loss.item():.4f}")

# Task 4 requirement: Test backpropagation
loss.backward()
print("Success: loss.backward() executed without errors!")


# 3. Simple mIoU Calculation
print("\n--- Step 3: Calculate mIoU ---")
# Get predicted class IDs (Batch, Height, Width)
preds_id = torch.argmax(fake_preds, dim=1)

iou_list = []

for class_id in range(NUM_CLASSES):
    pred_c = (preds_id == class_id)
    label_c = (fake_labels == class_id)

    # Intersection and Union
    intersection = (pred_c & label_c).sum().item()
    union = (pred_c | label_c).sum().item()

    # Avoid division by zero
    if union > 0:
        iou_list.append(intersection / union)

# Mean IoU
miou_score = sum(iou_list) / len(iou_list)
print(f"mIoU score: {miou_score:.4f}")

###Task 4: Map RGB Colors to Class IDs, Load Real CamVid Images, and Apply Synchronized Data Augmentation

###Task 5: Extract Pool3/Pool4 Feature Maps and Implement FCN-8s via Addition Fusion

###Task 6: Implement K-Fold Splitting, Weighted Cross-Entropy/Dice Loss, Model Saving, and Plot Loss Curves

###Task 7: Conduct Data Augmentation Ablation Study to Verify Overfitting Suppression

###Task 8: Conduct Segmentation Performance Comparison: FCN-32s vs. FCN-8s

###Task 9: Execute Full 5-Fold Cross-Validation on Optimal Configuration and Summarize Final Metrics

###Task 10: Write "Dataset Analysis and Preprocessing" Report Section and Clean Up Code Comments

###Task 11: Draw Network Architecture Diagram and Write "Network Structure" and "Difficulty Assessment" Sections

###Task 12: Compile Literature Benchmarks, Write "Results and Evaluation" Section, and Verify Generative_AI_Log.md

###Task 13: Jointly Verify Notebook and Submit Assignment